<div style="background:#0B1F4B;padding:22px 26px;border-radius:12px;border:3px solid #C9A227;font-family:'Times New Roman',serif">
<img src="https://raw.githubusercontent.com/drssnaiml1-cell/26UAMPC204-Artificial-Intelligence/main/assets/logo.png" alt="Sanjivani University" width="92" style="float:right;background:#FFFFFF;border:2px solid #C9A227;border-radius:8px;padding:3px;margin-left:14px">
<div style="color:#C9A227;font-size:14px;letter-spacing:2px;font-weight:bold">26UAMPC204 · ARTIFICIAL INTELLIGENCE · ASSIGNMENT 2 · UNIT II · CO204.2</div>
<div style="color:#FFFFFF;font-size:30px;font-weight:bold;margin-top:6px">Search Beyond the Textbook</div>
<div style="color:#F6EBC8;font-size:15px;margin-top:8px">Due: End of Week 6 · Name: ____________ · Roll No: ________</div>
<div style="clear:both"></div></div>

## 🧠 Part A — Think out of the box

Answer each in 250–400 words in the markdown cells below.

1. **[L5]** A* is optimal with an admissible heuristic. Give a real situation where you would deliberately use an inadmissible heuristic, and argue why the loss of optimality is acceptable.
2. **[L6]** Design a heuristic for finding the fastest route between two classrooms on your campus that accounts for stairs, lifts and crowds at bell time. Is it admissible? Prove or disprove it.
3. **[L4]** Why can a chess engine look 20 moves ahead but a cricket-captain AI cannot? Analyse the difference in branching factor, observability and stochasticity.
4. **[L6]** Invent a two-player game that is small enough to solve with Minimax but would be interesting for humans. Define its rules and predict who wins with perfect play.

**Answer A2.1:** _write here_

**Answer A2.2:** _write here_

**Answer A2.3:** _write here_

**Answer A2.4:** _write here_

## 💻 Part B — Open-ended coding

The starter code runs as-is and prints a baseline. Complete every `TODO`.

### B2.1 · Open-ended: heuristic design challenge

<div style="border-left:6px solid #1F3A7A;background:#EEF2F8;padding:12px 18px;border-radius:6px;font-family:'Times New Roman',serif;color:#1A202C">
<b style="color:#1F3A7A">Task</b><br>Invent at least two new admissible heuristics for the 8-puzzle (beyond misplaced tiles and Manhattan distance) and compare nodes expanded on 50 random solvable puzzles.</div>

In [1]:
import heapq, itertools, random
GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
def succ(s):
    i = s.index(0); r, c = divmod(i, 3)
    for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            j = nr * 3 + nc; t = list(s); t[i], t[j] = t[j], t[i]; yield tuple(t)
def manhattan(s):
    return sum(abs(k // 3 - (v - 1) // 3) + abs(k % 3 - (v - 1) % 3) for k, v in enumerate(s) if v)
def astar(start, h):
    tie = itertools.count(); fr = [(h(start), 0, next(tie), start)]; g = {start: 0}; n = 0
    while fr:
        f, gs, _, s = heapq.heappop(fr); n += 1
        if s == GOAL: return gs, n
        for t in succ(s):
            if gs + 1 < g.get(t, 99): g[t] = gs + 1; heapq.heappush(fr, (gs + 1 + h(t), gs + 1, next(tie), t))
def scramble(k, seed):
    rnd = random.Random(seed); s = GOAL
    for _ in range(k): s = rnd.choice(list(succ(s)))
    return s
puzzles = [scramble(30, i) for i in range(10)]
print("Manhattan: mean nodes expanded =", sum(astar(p, manhattan)[1] for p in puzzles) / len(puzzles))
# TODO 1: implement 'linear conflict' (Manhattan + 2 per pair of tiles in the same row/col in reverse order).
# TODO 2: invent your own heuristic and prove it is admissible.
# TODO 3: run on 50 puzzles and plot nodes expanded for each heuristic.

Manhattan: mean nodes expanded = 49.7


In [2]:
# Your solution for B2.1


### B2.2 · Open-ended: local search on a real optimisation problem

<div style="border-left:6px solid #1F3A7A;background:#EEF2F8;padding:12px 18px;border-radius:6px;font-family:'Times New Roman',serif;color:#1A202C">
<b style="color:#1F3A7A">Task</b><br>Choose a real optimisation task (seating plan, delivery tour, playlist ordering) and solve it with hill climbing and simulated annealing. Report how often each reaches the best value you found.</div>

In [3]:
import math, random
random.seed(0)
cities = [(random.random(), random.random()) for _ in range(25)]
def length(t): return sum(math.dist(cities[t[i]], cities[t[i - 1]]) for i in range(len(t)))
def two_opt_neighbour(t):
    i, j = sorted(random.sample(range(len(t)), 2)); return t[:i] + t[i:j][::-1] + t[j:]
def anneal(T=1.0, cool=0.995, steps=20000):
    cur = list(range(len(cities))); random.shuffle(cur); best = cur
    for _ in range(steps):
        nxt = two_opt_neighbour(cur); d = length(nxt) - length(cur)
        if d < 0 or random.random() < math.exp(-d / T): cur = nxt
        if length(cur) < length(best): best = cur
        T *= cool
    return length(best)
print("Simulated annealing tour length:", round(anneal(), 3))
# TODO: implement hill climbing with random restarts, compare over 20 seeds,
#       and try a different cooling schedule. Which matters more: schedule or neighbourhood?

Simulated annealing tour length: 4.184


In [4]:
# Your solution for B2.2


### B2.3 · Open-ended: build and break a game AI

<div style="border-left:6px solid #1F3A7A;background:#EEF2F8;padding:12px 18px;border-radius:6px;font-family:'Times New Roman',serif;color:#1A202C">
<b style="color:#1F3A7A">Task</b><br>Write a Minimax + alpha–beta agent for Connect-Three on a 4×4 board (or your own game). Then write an evaluation function for a depth-limited version and show a position where it blunders.</div>

In [5]:
import math
ROWS, COLS, WIN = 4, 4, 3
def moves(b): return [c for c in range(COLS) if b[0][c] == "."]
def drop(b, c, p):
    nb = [row[:] for row in b]
    for r in range(ROWS - 1, -1, -1):
        if nb[r][c] == ".": nb[r][c] = p; return nb
def winner(b):
    for r in range(ROWS):
        for c in range(COLS):
            for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
                cells = [(r + k * dr, c + k * dc) for k in range(WIN)]
                if all(0 <= x < ROWS and 0 <= y < COLS for x, y in cells):
                    v = {b[x][y] for x, y in cells}
                    if len(v) == 1 and "." not in v: return v.pop()
    return None
def ab(b, p, a=-math.inf, be=math.inf):
    w = winner(b)
    if w: return 1 if w == "X" else -1
    if not moves(b): return 0
    best = -math.inf if p == "X" else math.inf
    for c in moves(b):
        v = ab(drop(b, c, p), "O" if p == "X" else "X", a, be)
        if p == "X": best = max(best, v); a = max(a, v)
        else: best = min(best, v); be = min(be, v)
        if a >= be: break
    return best
empty = [["."] * COLS for _ in range(ROWS)]
print("Value of the empty 4x4 Connect-Three board for X:", ab(empty, "X"))
# TODO: add a depth limit + evaluation function, measure nodes searched,
#       and find a position where the depth-limited agent makes a wrong move.

Value of the empty 4x4 Connect-Three board for X: 1


In [6]:
# Your solution for B2.3


## 📋 Reflection

What surprised you? What would you do with one more week? How did you use AI tools, if at all?